# AI Agents — Maintenance Agent & Alert Agent

This notebook sits **after the Smart Twin + IoT Simulation**.

The Smart Twin predicts machine condition. The agents turn those predictions into operational decisions.

```text
IoT Simulation
      ↓
Smart Twin
      ↓
┌──────────────────────────────┐
│        Agent Layer           │
│                              │
│  Maintenance Agent           │
│  Alert Agent                 │
└──────────────┬───────────────┘
               ↓
      Action + Alert + Priority
```

This first implementation is deterministic and auditable. It does not require an external LLM/API, which makes it reliable for the capstone demo.


In [1]:
# ============================================
# STEP 1: IMPORTS & LOAD SMART TWIN RESULTS
# ============================================

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

results_path = "../data/processed/iot_smart_twin_results.csv"

smart_twin_df = pd.read_csv(results_path)

print("Smart Twin results loaded.")
print("Shape:", smart_twin_df.shape)

display(smart_twin_df.head())


Smart Twin results loaded.
Shape: (1930, 24)


,Timestamp,Machine_ID,Machine_Type,Cycle,Temperature_C,Vibration_mm_s,Pressure_bar,Rotational_Speed_RPM,Load_Fraction,Torque_Nm,...,Anomaly_Score,Simulation_Fault,failure_prediction,failure_probability,failure_risk,predicted_rul_cycles,rul_status,overall_status,maintenance_action,health_score
0,2026-08-01 08:00:00,CNC_001,CNC_Mill,1,69.030303,0.213912,2.113994,1517.415010,0.656548,46.756227,...,0.1342,BEARING_WEAR,0,0.0001,LOW,256.45,HEALTHY,HEALTHY,NORMAL OPERATION,99.99
1,2026-08-01 08:01:00,CNC_001,CNC_Mill,2,69.089661,0.167449,2.139051,1517.500142,0.594057,46.788014,...,0.2990,BEARING_WEAR,0,0.0001,LOW,262.11,HEALTHY,HEALTHY,NORMAL OPERATION,99.99
2,2026-08-01 08:02:00,CNC_001,CNC_Mill,3,69.030239,0.246411,2.120718,1522.832970,0.628487,46.321709,...,0.2269,BEARING_WEAR,0,0.0001,LOW,256.96,HEALTHY,HEALTHY,NORMAL OPERATION,99.99
3,2026-08-01 08:03:00,CNC_001,CNC_Mill,4,69.040893,0.205632,2.114275,1534.355835,0.695731,48.495093,...,0.0469,BEARING_WEAR,0,0.0001,LOW,256.90,HEALTHY,HEALTHY,NORMAL OPERATION,99.99
4,2026-08-01 08:04:00,CNC_001,CNC_Mill,5,68.651431,0.200355,2.085129,1515.119912,0.593584,48.107525,...,0.2841,BEARING_WEAR,0,0.0001,LOW,262.69,HEALTHY,HEALTHY,NORMAL OPERATION,99.99


In [2]:
# ============================================
# STEP 2: VERIFY REQUIRED AGENT INPUTS
# ============================================

required_columns = [
    "Machine_ID",
    "Machine_Type",
    "Cycle",
    "Simulation_Fault",
    "failure_probability",
    "failure_risk",
    "predicted_rul_cycles",
    "rul_status",
    "overall_status",
    "maintenance_action",
    "health_score"
]

missing = [
    c for c in required_columns
    if c not in smart_twin_df.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

print("All required Smart Twin fields are available.")


All required Smart Twin fields are available.


## Agent 1 — Maintenance Agent

The Maintenance Agent answers:

> **What maintenance action should be taken, and how urgent is it?**

It considers failure risk, predicted RUL, machine status and health score.


In [3]:
# ============================================
# STEP 3: MAINTENANCE AGENT
# ============================================

def maintenance_agent(row):
    failure_risk = row["failure_risk"]
    rul = float(row["predicted_rul_cycles"])
    status = row["overall_status"]
    health = float(row["health_score"])

    if failure_risk == "HIGH" or rul <= 20 or status == "CRITICAL":
        priority = "URGENT"
        action = "IMMEDIATE MAINTENANCE"
        reason = (
            "High failure risk and/or critically low remaining "
            "useful life detected."
        )

    elif failure_risk == "MEDIUM" or rul <= 50 or status == "WARNING":
        priority = "HIGH"
        action = "SCHEDULE INSPECTION"
        reason = (
            "Machine degradation is significant and should be "
            "inspected before the condition worsens."
        )

    else:
        priority = "ROUTINE"
        action = "NORMAL OPERATION"
        reason = (
            "Machine condition is currently within the healthy "
            "operating range."
        )

    if health < 30:
        priority = "URGENT"
        action = "IMMEDIATE MAINTENANCE"
        reason += " Health score is also critically low."

    return {
        "maintenance_priority": priority,
        "maintenance_decision": action,
        "maintenance_reason": reason
    }

print("Maintenance Agent ready.")


Maintenance Agent ready.


In [4]:
# ============================================
# STEP 4: TEST MAINTENANCE AGENT
# ============================================

sample = smart_twin_df.iloc[0]

maintenance_result = maintenance_agent(sample)

print("MAINTENANCE AGENT")
print("=================")
print("Machine:", sample["Machine_ID"])
print("RUL:", sample["predicted_rul_cycles"])
print("Failure Risk:", sample["failure_risk"])
print("Health Score:", sample["health_score"])

for key, value in maintenance_result.items():
    print(f"{key}: {value}")


MAINTENANCE AGENT
Machine: CNC_001
RUL: 256.45
Failure Risk: LOW
Health Score: 99.99
maintenance_priority: ROUTINE
maintenance_decision: NORMAL OPERATION
maintenance_reason: Machine condition is currently within the healthy operating range.


## Agent 2 — Alert Agent

The Alert Agent answers:

> **Does this machine require an alert, and what severity should be sent to the operator?**

It separates routine machine states from actionable events and creates a concise alert message.


In [5]:
# ============================================
# STEP 5: ALERT AGENT
# ============================================

def alert_agent(row, maintenance_result):
    machine_id = row["Machine_ID"]
    machine_type = row["Machine_Type"]
    cycle = row["Cycle"]

    risk = row["failure_risk"]
    rul = float(row["predicted_rul_cycles"])
    status = row["overall_status"]
    probability = float(row["failure_probability"])
    action = maintenance_result["maintenance_decision"]

    if (
        risk == "HIGH"
        or rul <= 20
        or status == "CRITICAL"
    ):
        alert_level = "CRITICAL"
        alert_required = True
        message = (
            f"CRITICAL ALERT: {machine_id} ({machine_type}) "
            f"requires immediate attention. "
            f"Failure probability={probability:.2%}, "
            f"predicted RUL={rul:.2f} cycles. "
            f"Recommended action: {action}."
        )

    elif (
        risk == "MEDIUM"
        or rul <= 50
        or status == "WARNING"
    ):
        alert_level = "WARNING"
        alert_required = True
        message = (
            f"WARNING: {machine_id} ({machine_type}) "
            f"shows degradation. "
            f"Failure probability={probability:.2%}, "
            f"predicted RUL={rul:.2f} cycles. "
            f"Recommended action: {action}."
        )

    else:
        alert_level = "INFO"
        alert_required = False
        message = (
            f"INFO: {machine_id} ({machine_type}) "
            f"is operating normally at cycle {cycle}."
        )

    return {
        "alert_required": alert_required,
        "alert_level": alert_level,
        "alert_message": message
    }

print("Alert Agent ready.")


Alert Agent ready.


In [6]:
# ============================================
# STEP 6: TEST ALERT AGENT
# ============================================

alert_result = alert_agent(
    sample,
    maintenance_result
)

print("ALERT AGENT")
print("===========")

for key, value in alert_result.items():
    print(f"{key}: {value}")


ALERT AGENT
alert_required: False
alert_level: INFO
alert_message: INFO: CNC_001 (CNC_Mill) is operating normally at cycle 1.


In [7]:
# ============================================
# STEP 7: COMBINED AGENT DECISION
# ============================================

def run_agents(row):
    maintenance_result = maintenance_agent(row)
    alert_result = alert_agent(
        row,
        maintenance_result
    )

    return {
        **maintenance_result,
        **alert_result
    }

print("Combined agent pipeline ready.")


Combined agent pipeline ready.


In [8]:
# ============================================
# STEP 8: RUN BOTH AGENTS ON ALL READINGS
# ============================================

agent_results = []

for _, row in smart_twin_df.iterrows():
    result = run_agents(row)

    result["Machine_ID"] = row["Machine_ID"]
    result["Machine_Type"] = row["Machine_Type"]
    result["Cycle"] = row["Cycle"]
    result["Timestamp"] = row["Timestamp"]
    result["Simulation_Fault"] = row["Simulation_Fault"]

    result["failure_probability"] = row[
        "failure_probability"
    ]
    result["predicted_rul_cycles"] = row[
        "predicted_rul_cycles"
    ]
    result["overall_status"] = row[
        "overall_status"
    ]

    agent_results.append(result)

agent_df = pd.DataFrame(agent_results)

print("Agent decisions generated.")
print("Shape:", agent_df.shape)

display(agent_df.head(10))


Agent decisions generated.
Shape: (1930, 14)


,maintenance_priority,maintenance_decision,maintenance_reason,alert_required,alert_level,alert_message,Machine_ID,Machine_Type,Cycle,Timestamp,Simulation_Fault,failure_probability,predicted_rul_cycles,overall_status
0,ROUTINE,NORMAL OPERATION,Machine condition is currently within the heal...,False,INFO,INFO: CNC_001 (CNC_Mill) is operating normally...,CNC_001,CNC_Mill,1,2026-08-01 08:00:00,BEARING_WEAR,0.0001,256.45,HEALTHY
1,ROUTINE,NORMAL OPERATION,Machine condition is currently within the heal...,False,INFO,INFO: CNC_001 (CNC_Mill) is operating normally...,CNC_001,CNC_Mill,2,2026-08-01 08:01:00,BEARING_WEAR,0.0001,262.11,HEALTHY
2,ROUTINE,NORMAL OPERATION,Machine condition is currently within the heal...,False,INFO,INFO: CNC_001 (CNC_Mill) is operating normally...,CNC_001,CNC_Mill,3,2026-08-01 08:02:00,BEARING_WEAR,0.0001,256.96,HEALTHY
3,ROUTINE,NORMAL OPERATION,Machine condition is currently within the heal...,False,INFO,INFO: CNC_001 (CNC_Mill) is operating normally...,CNC_001,CNC_Mill,4,2026-08-01 08:03:00,BEARING_WEAR,0.0001,256.90,HEALTHY
4,ROUTINE,NORMAL OPERATION,Machine condition is currently within the heal...,False,INFO,INFO: CNC_001 (CNC_Mill) is operating normally...,CNC_001,CNC_Mill,5,2026-08-01 08:04:00,BEARING_WEAR,0.0001,262.69,HEALTHY
5,ROUTINE,NORMAL OPERATION,Machine condition is currently within the heal...,False,INFO,INFO: CNC_001 (CNC_Mill) is operating normally...,CNC_001,CNC_Mill,6,2026-08-01 08:05:00,BEARING_WEAR,0.0001,253.78,HEALTHY
6,ROUTINE,NORMAL OPERATION,Machine condition is currently within the heal...,False,INFO,INFO: CNC_001 (CNC_Mill) is operating normally...,CNC_001,CNC_Mill,7,2026-08-01 08:06:00,BEARING_WEAR,0.0001,253.29,HEALTHY
7,ROUTINE,NORMAL OPERATION,Machine condition is currently within the heal...,False,INFO,INFO: CNC_001 (CNC_Mill) is operating normally...,CNC_001,CNC_Mill,8,2026-08-01 08:07:00,BEARING_WEAR,0.0001,255.17,HEALTHY
8,ROUTINE,NORMAL OPERATION,Machine condition is currently within the heal...,False,INFO,INFO: CNC_001 (CNC_Mill) is operating normally...,CNC_001,CNC_Mill,9,2026-08-01 08:08:00,BEARING_WEAR,0.0001,251.45,HEALTHY
9,ROUTINE,NORMAL OPERATION,Machine condition is currently within the heal...,False,INFO,INFO: CNC_001 (CNC_Mill) is operating normally...,CNC_001,CNC_Mill,10,2026-08-01 08:09:00,BEARING_WEAR,0.0001,250.34,HEALTHY


In [9]:
# ============================================
# STEP 9: CRITICAL ALERTS
# ============================================

critical_alerts = agent_df[
    agent_df["alert_level"] == "CRITICAL"
].copy()

critical_alerts = critical_alerts.sort_values(
    by=[
        "failure_probability",
        "predicted_rul_cycles"
    ],
    ascending=[
        False,
        True
    ]
)

print("Critical alerts:", len(critical_alerts))

display(
    critical_alerts[
        [
            "Timestamp",
            "Machine_ID",
            "Machine_Type",
            "Cycle",
            "Simulation_Fault",
            "failure_probability",
            "predicted_rul_cycles",
            "maintenance_priority",
            "maintenance_decision",
            "alert_message"
        ]
    ].head(20)
)


Critical alerts: 286


,Timestamp,Machine_ID,Machine_Type,Cycle,Simulation_Fault,failure_probability,predicted_rul_cycles,maintenance_priority,maintenance_decision,alert_message
1406,2026-08-01 13:06:00,TURB_005,Turbine,307,MISALIGNMENT,0.9983,2.88,URGENT,IMMEDIATE MAINTENANCE,CRITICAL ALERT: TURB_005 (Turbine) requires im...
1704,2026-08-01 12:34:00,PRESS_006,Hydraulic_Press,275,PRESSURE,0.9981,1.63,URGENT,IMMEDIATE MAINTENANCE,CRITICAL ALERT: PRESS_006 (Hydraulic_Press) re...
1695,2026-08-01 12:25:00,PRESS_006,Hydraulic_Press,266,PRESSURE,0.9981,2.30,URGENT,IMMEDIATE MAINTENANCE,CRITICAL ALERT: PRESS_006 (Hydraulic_Press) re...
838,2026-08-01 12:28:00,COMP_003,Compressor,269,VIBRATION,0.9980,2.29,URGENT,IMMEDIATE MAINTENANCE,CRITICAL ALERT: COMP_003 (Compressor) requires...
568,2026-08-01 13:08:00,PUMP_002,Hydraulic_Pump,309,OVERHEATING,0.9980,2.53,URGENT,IMMEDIATE MAINTENANCE,CRITICAL ALERT: PUMP_002 (Hydraulic_Pump) requ...
1094,2026-08-01 11:54:00,MOTOR_004,Electric_Motor,235,ELECTRICAL,0.9979,2.71,URGENT,IMMEDIATE MAINTENANCE,CRITICAL ALERT: MOTOR_004 (Electric_Motor) req...
1694,2026-08-01 12:24:00,PRESS_006,Hydraulic_Press,265,PRESSURE,0.9979,3.03,URGENT,IMMEDIATE MAINTENANCE,CRITICAL ALERT: PRESS_006 (Hydraulic_Press) re...
1401,2026-08-01 13:01:00,TURB_005,Turbine,302,MISALIGNMENT,0.9979,3.37,URGENT,IMMEDIATE MAINTENANCE,CRITICAL ALERT: TURB_005 (Turbine) requires im...
1702,2026-08-01 12:32:00,PRESS_006,Hydraulic_Press,273,PRESSURE,0.9979,3.41,URGENT,IMMEDIATE MAINTENANCE,CRITICAL ALERT: PRESS_006 (Hydraulic_Press) re...
1426,2026-08-01 13:26:00,TURB_005,Turbine,327,MISALIGNMENT,0.9978,1.82,URGENT,IMMEDIATE MAINTENANCE,CRITICAL ALERT: TURB_005 (Turbine) requires im...


In [10]:
# ============================================
# STEP 10: MAINTENANCE PRIORITY DISTRIBUTION
# ============================================

print("Maintenance priority distribution:")
display(
    agent_df["maintenance_priority"]
    .value_counts()
)

print("\nAlert distribution:")
display(
    agent_df["alert_level"]
    .value_counts()
)


Maintenance priority distribution:


maintenance_priority
ROUTINE    1448
URGENT      286
HIGH        196
Name: count, dtype: int64


Alert distribution:


alert_level
INFO        1448
CRITICAL     286
WARNING      196
Name: count, dtype: int64

In [11]:
# ============================================
# STEP 11: LATEST DECISION FOR EACH MACHINE
# ============================================

latest_agent_state = (
    agent_df
    .sort_values("Cycle")
    .groupby("Machine_ID", as_index=False)
    .tail(1)
    .sort_values(
        by=[
            "maintenance_priority",
            "predicted_rul_cycles"
        ],
        ascending=[
            True,
            True
        ]
    )
)

display(
    latest_agent_state[
        [
            "Machine_ID",
            "Machine_Type",
            "Cycle",
            "Simulation_Fault",
            "failure_probability",
            "predicted_rul_cycles",
            "overall_status",
            "maintenance_priority",
            "maintenance_decision",
            "alert_level",
            "alert_required"
        ]
    ]
)


,Machine_ID,Machine_Type,Cycle,Simulation_Fault,failure_probability,predicted_rul_cycles,overall_status,maintenance_priority,maintenance_decision,alert_level,alert_required
1099,MOTOR_004,Electric_Motor,240,ELECTRICAL,0.9968,1.33,CRITICAL,URGENT,IMMEDIATE MAINTENANCE,CRITICAL,True
1704,PRESS_006,Hydraulic_Press,275,PRESSURE,0.9981,1.63,CRITICAL,URGENT,IMMEDIATE MAINTENANCE,CRITICAL,True
1429,TURB_005,Turbine,330,MISALIGNMENT,0.9967,1.64,CRITICAL,URGENT,IMMEDIATE MAINTENANCE,CRITICAL,True
1929,ROBOT_007,Industrial_Robot,225,ACTUATOR_DEGRADATION,0.9941,1.86,CRITICAL,URGENT,IMMEDIATE MAINTENANCE,CRITICAL,True
259,CNC_001,CNC_Mill,260,BEARING_WEAR,0.9939,2.06,CRITICAL,URGENT,IMMEDIATE MAINTENANCE,CRITICAL,True
569,PUMP_002,Hydraulic_Pump,310,OVERHEATING,0.9977,2.12,CRITICAL,URGENT,IMMEDIATE MAINTENANCE,CRITICAL,True
859,COMP_003,Compressor,290,VIBRATION,0.9887,3.89,CRITICAL,URGENT,IMMEDIATE MAINTENANCE,CRITICAL,True


In [12]:
# ============================================
# STEP 12: FINAL AGENT REPORT
# ============================================

urgent = latest_agent_state[
    latest_agent_state["maintenance_priority"] == "URGENT"
]

high = latest_agent_state[
    latest_agent_state["maintenance_priority"] == "HIGH"
]

routine = latest_agent_state[
    latest_agent_state["maintenance_priority"] == "ROUTINE"
]

print("AI AGENT MAINTENANCE REPORT")
print("===========================")
print("Machines requiring URGENT maintenance:", len(urgent))
print("Machines requiring inspection:", len(high))
print("Machines operating normally:", len(routine))

if len(urgent) > 0:
    print("\nURGENT MACHINES:")
    for _, row in urgent.iterrows():
        print(
            f"- {row['Machine_ID']} | "
            f"{row['Machine_Type']} | "
            f"RUL={row['predicted_rul_cycles']:.2f} | "
            f"Failure={row['failure_probability']:.2%}"
        )


AI AGENT MAINTENANCE REPORT
Machines requiring URGENT maintenance: 7
Machines requiring inspection: 0
Machines operating normally: 0

URGENT MACHINES:
- MOTOR_004 | Electric_Motor | RUL=1.33 | Failure=99.68%
- PRESS_006 | Hydraulic_Press | RUL=1.63 | Failure=99.81%
- TURB_005 | Turbine | RUL=1.64 | Failure=99.67%
- ROBOT_007 | Industrial_Robot | RUL=1.86 | Failure=99.41%
- CNC_001 | CNC_Mill | RUL=2.06 | Failure=99.39%
- PUMP_002 | Hydraulic_Pump | RUL=2.12 | Failure=99.77%
- COMP_003 | Compressor | RUL=3.89 | Failure=98.87%


In [13]:
# ============================================
# STEP 13: SAVE AGENT OUTPUT
# ============================================

output_path = "../data/processed/ai_agent_results.csv"

agent_df.to_csv(
    output_path,
    index=False
)

print("Agent results saved.")
print("Path:", output_path)


Agent results saved.
Path: ../data/processed/ai_agent_results.csv


## Completed Agent Architecture

```text
IoT Simulation
      ↓
Smart Twin
      ↓
 ┌────┴─────┐
 ↓          ↓
Maintenance Agent
             Alert Agent
 ↓          ↓
Action      Operator Alert
 └────┬─────┘
      ↓
AI Agent Result
```

### Next phase

The next step is **Memory + RAG**.

Memory will preserve previous machine events and maintenance actions.

RAG will provide maintenance knowledge so the agents can explain *why* a condition is dangerous and suggest relevant maintenance procedures.
